<a href="https://colab.research.google.com/github/GDjkhp/rvc-for-ai-music-covers/blob/main/RVC_Inference_by_GDjkhp.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

WTFPL

In [ ]:
#@title 0. Mount Google Drive (Optional)
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
#@title 1. Install RVC
import shlex, shutil, subprocess, sys, time
from pathlib import Path

RVC_DIR = Path("/content/RVC")
VENV = Path("/content/rvc-venv")
HF = "https://huggingface.co/lj1995/%s%s%s/resolve/main" % ("Voice", "Conversion", "WebUI")

# Inference-only dependencies. RVC's requirements.txt also installs gradio, tensorboard,
# matplotlib, scikit-learn, onnxruntime-gpu, ... which the inference CLI never imports.
# torch is pinned here so it is downloaded exactly once.
REQUIREMENTS = """\
numpy==1.23.5
scipy
numba
llvmlite
librosa==0.9.1
resampy>=0.4.2
soundfile>=0.12.1
ffmpeg-python>=0.2.0
av
praat-parselmouth>=0.4.2
pyworld==0.3.2
torch==2.5.1
torchaudio==2.5.1
torchcrepe==0.0.20
fairseq==0.12.2
faiss-cpu
joblib>=1.1.0
tqdm
python-dotenv
setuptools<81
"""  # setuptools<81: librosa 0.9.1, pyworld and torchcrepe import pkg_resources

def run(cmd):
    print("$", shlex.join(cmd))
    subprocess.run(cmd, check=True)

t0 = time.time()

# uv fetches Python 3.10 in seconds and installs packages in parallel.
run([sys.executable, "-m", "pip", "install", "-q", "uv"])
uv = [sys.executable, "-m", "uv"]

if not RVC_DIR.exists():
    run(["git", "clone", "--depth", "1", "https://github.com/RVC-Project/Retrieval-based-%s-%s-%s" % ("Voice", "Conversion", "WebUI"), str(RVC_DIR)])
Path("/content/dataset").mkdir(exist_ok=True)

# Start the model downloads now so they run while the packages install.
downloads = []
for dest, url in {
    RVC_DIR / "assets/hubert/hubert_base.pt": f"{HF}/hubert_base.pt",
    RVC_DIR / "assets/rmvpe/rmvpe.pt": f"{HF}/rmvpe.pt",
}.items():
    if dest.exists():
        continue
    dest.parent.mkdir(parents=True, exist_ok=True)
    part = dest.with_name(dest.name + ".part")
    downloads.append((subprocess.Popen(["curl", "-sSfL", "--retry", "3", "-C", "-", "-o", str(part), url]), part, dest))

run(uv + ["venv", "--python", "3.10", "--allow-existing", str(VENV)])
(VENV / "requirements.txt").write_text(REQUIREMENTS)
run(uv + ["pip", "install", "--python", str(VENV / "bin/python"), "-r", str(VENV / "requirements.txt")])

for proc, part, dest in downloads:
    if proc.wait() != 0:
        raise RuntimeError(f"Failed to download {dest.name}. Re-run this cell to resume.")
    part.rename(dest)

if not shutil.which("nvidia-smi"):
    print("WARNING: no GPU detected. Use Runtime > Change runtime type > T4 GPU.")
print(f"RVC is ready ({time.time() - t0:.0f}s)")

In [ ]:
#@title 2. Install Demucs (Optional)
# Installed into the RVC environment. The PyPI release doesn't pin torch, so RVC keeps its torch version.
# (The git version pins torchaudio<2.1, which would downgrade torch.)
!uv pip install --python /content/rvc-venv/bin/python demucs

In [ ]:
#@title 3. Demix Audio (Optional)
import os
working_dir = "/content/dataset" # @param {"type":"string"}
audio = "【ハク。Cover企画】MONO NO AWARE ＂かむかもしかもにどもかも！＂.mp3" # @param {"type":"string"}
#@markdown ### Demucs model (-n)
#@markdown - `htdemucs`: default, 4 stems (vocals, drums, bass, other)
#@markdown - `htdemucs_ft`: fine-tuned, better quality but ~4x slower
#@markdown - `htdemucs_6s`: 6 stems (adds guitar and piano, piano is experimental)
#@markdown - `hdemucs_mmi`: Hybrid Demucs v3 trained on extra data
#@markdown - `mdx`, `mdx_extra`: MDX challenge winners (`mdx_extra` is the stronger one)
#@markdown - `mdx_q`, `mdx_extra_q`: quantized versions, smaller and faster, slightly lower quality
demucs_model = "htdemucs" # @param ["htdemucs", "htdemucs_ft", "htdemucs_6s", "hdemucs_mmi", "mdx", "mdx_extra", "mdx_q", "mdx_extra_q"]
!/content/rvc-venv/bin/python -m demucs.separate -n {demucs_model} "{working_dir}/{audio}" -o "{working_dir}"
stem_path = f"{working_dir}/{demucs_model}/{os.path.splitext(audio)[0]}"
# Copy every stem in one command (works for 4-stem and 6-stem models)
!cp -v "{stem_path}"/* "{working_dir}/"

In [ ]:
#@title 4. Infer
# Claude rewrote everything for simplicity (ongoing)
import os
import shutil
import requests
from pathlib import Path
import zipfile
import subprocess
from urllib.parse import unquote

RVC_PY = "/content/rvc-venv/bin/python"  # created by step 1

#@markdown # Settings
#@markdown ### Input/Output
#@markdown - Set current working directory. Input files must be here. Output files will be generated here.
working_dir = "/content/dataset" # @param {"type":"string"}
input_audio = "vocals.wav" # @param {"type":"string"}
output_audio = "miku_output.wav" # @param {"type":"string"}
#@markdown ### Model
#@markdown - Only `Hugging Face` links are supported. **MAKE SURE** it's an `RVC` model and **NOT** a `so-vits-svc` model. It will return an error at download.
#@markdown - `RVC` models contain `*.pth and *.index` files while `so-vits-svc` models contain `config.json and *.pth` files.
model_url = "https://huggingface.co/aple/HatsuneMikuRVC/resolve/main/miku_default_rvc.zip" # @param {"type":"string"}
decoded_url = unquote(model_url)
filename = os.path.basename(decoded_url)
model_name = os.path.splitext(filename)[0]
#@markdown ### Pitch
#@markdown - **Female -> Male : -12**
#@markdown - **Male -> Female : 12**
pitch = 0 # @param {"type":"slider","min":-12,"max":12,"step":1}
#@markdown ---
#@markdown ### Advanced
f0_method = "rmvpe" # @param ["rmvpe", "pm", "harvest"]
index_rate = 0.5 # @param {type:"slider", min:0, max:1, step:0.01}
volume_normalization = 0 #@param {type:"slider", min:0, max:1, step:0.01}
consonant_protection = 0.5 #@param {type:"slider", min:0, max:1, step:0.01}

class RVCInference:
    def __init__(self, working_dir: str | Path):
        """
        Initialize RVC inference with a configurable working directory.

        Args:
            working_dir (str | Path): Path to the working directory for input/output files
        """
        self.input_files_dir = Path(working_dir)
        self.models = {}
        self._setup_directory_structure()

    def _setup_directory_structure(self):
        """Create necessary directory structure."""
        self.input_files_dir.mkdir(parents=True, exist_ok=True)

    def verify_files(self, paths: dict) -> None:
        """Verify all required input files exist."""
        # Verify all paths except output_path
        for key, path in paths.items():
            if key != 'output_path':  # Skip output path verification
                if not os.path.exists(path):
                    raise FileNotFoundError(f"{key} not found at: {path}")
                print(f"Found {key} at: {path}")

        # Verify output directory exists
        output_dir = os.path.dirname(paths['output_path'])
        if not os.path.exists(output_dir):
            raise FileNotFoundError(f"Output directory not found at: {output_dir}")
        print(f"Output will be saved to: {paths['output_path']}")

    def verify_input_file(self, input_filename: str) -> str:
        """Verify input file exists and return its full path."""
        input_path = self.input_files_dir / input_filename
        if not input_path.exists():
            raise FileNotFoundError(f"Input file not found: {input_path}")
        return str(input_path)

    def download_and_extract_model(self, url: str, model_name: str) -> bool:
        """Download and extract model from URL to appropriate directory."""
        model_dir = self.input_files_dir / model_name
        model_dir.mkdir(exist_ok=True)

        try:
            # Download zip file
            print(f"Downloading model from {url}")
            response = requests.get(url, stream=True)
            response.raise_for_status()

            # Save zip file
            zip_path = model_dir / "temp.zip"
            with open(zip_path, 'wb') as f:
                for chunk in response.iter_content(chunk_size=8192):
                    f.write(chunk)

            # Extract zip file
            print(f"Extracting model to {model_dir}")
            with zipfile.ZipFile(zip_path, 'r') as zip_ref:
                zip_ref.extractall(model_dir)

            # Clean up zip file
            zip_path.unlink()

            # Register model
            self._register_model(model_name, model_dir)
            return True

        except Exception as e:
            print(f"Error downloading/extracting model: {e}")
            if model_dir.exists():
                shutil.rmtree(model_dir)
            return False

    def _register_model(self, model_name: str, model_dir: Path):
        """Register model paths after download/extraction."""
        # Find .pth and .index files
        pth_files = list(model_dir.rglob("*.pth"))
        index_files = list(model_dir.rglob("*.index"))

        if not pth_files or not index_files:
            raise FileNotFoundError(f"Missing required files in {model_dir}")

        self.models[model_name] = {
            'pth': str(pth_files[0]),
            'index': str(index_files[0])
        }

    def setup_paths(self, model_name: str, input_filename: str) -> dict:
        """Setup all necessary paths for inference."""
        if model_name not in self.models:
            raise ValueError(f"Model {model_name} not found. Please download it first.")

        return {
            'model_path': self.models[model_name]['pth'],
            'index_path': self.models[model_name]['index'],
            'input_path': self.verify_input_file(input_filename),
            'output_path': str(self.input_files_dir)
        }

    def set_output_filename(self, filename: str):
        """Set custom output filename and remove if exists."""
        output_path = self.input_files_dir / filename
        if output_path.exists():
            output_path.unlink()
        return str(output_path)

    def run_inference(self, model_name: str, input_filename: str = "vocals.wav",
                     output_filename: str = "output.wav", pitch: int = 0,
                     f0_method: str = "rmvpe", index_rate: float = 0.5,
                     volume_norm: float = 0, consonant_protect: float = 0.5) -> str:
        """Run the RVC inference pipeline."""
        if not os.path.exists(RVC_PY):
            raise RuntimeError("RVC is not installed. Run step 1 (Install RVC) first.")

        # Setup and verify paths
        paths = self.setup_paths(model_name, input_filename)
        paths['output_path'] = self.set_output_filename(output_filename)
        self.verify_files(paths)

        # Set environment variables
        os.environ['index_root'] = os.path.dirname(paths['index_path'])
        os.environ['weight_root'] = os.path.dirname(paths['model_path'])

        # Prepare command
        input = paths['input_path']
        index = os.path.basename(paths['index_path'])
        output = paths['output_path']
        model = os.path.basename(paths['model_path'])

        # Run command
        os.chdir("/content/RVC")
        # The CLI lives at tools/infer_cli.py in some RVC layouts and tools/cmd/infer_cli.py in others.
        cli = next((p for p in ("tools/infer_cli.py", "tools/cmd/infer_cli.py") if os.path.exists(p)), None)
        if cli is None:
            raise FileNotFoundError("infer_cli.py not found under /content/RVC/tools")
        command = [
            RVC_PY, cli,
            "--f0up_key", str(pitch),
            "--input_path", input,
            "--index_path", index,
            "--f0method", f0_method,
            "--opt_path", output,
            "--model_name", model,
            "--index_rate", str(index_rate),
            "--device", "cuda:0",
            "--is_half", "True",
            "--filter_radius", "3",
            "--resample_sr", "0",
            "--rms_mix_rate", str(volume_norm),
            "--protect", str(consonant_protect)
        ]
        result = subprocess.run(command, capture_output=True, text=True)

        if result.returncode == 0:
            print(result.stdout)
        else:
            print("Error occurred:", result.stderr)

        return paths['output_path']

def run_voice_conversion(
    model_url: str,
    model_name: str,
    working_dir: str | Path = "/content/workingdir",
    input_filename: str = "vocals.wav",
    output_filename: str = "output.wav",
    pitch: int = 0,
    f0_method: str = "rmvpe",
    index_rate: float = 0.5,
    volume_norm: float = 0,
    consonant_protect: float = 0.5
):
    """
    Main function to run voice conversion with configurable working directory.

    Args:
        model_url (str): URL to download the model from
        model_name (str): Name to identify the model
        working_dir (str | Path): Directory for input/output files and models
        input_filename (str): Name of the input audio file
        output_filename (str): Name for the output audio file
        pitch (int): Pitch adjustment value (-12 to 12)
    """
    rvc = RVCInference(working_dir=working_dir)

    # Download and extract model if not already present
    if model_name not in rvc.models:
        success = rvc.download_and_extract_model(model_url, model_name)
        if not success:
            raise RuntimeError(f"Failed to download/extract model {model_name}")

    # Run inference with default parameters
    output_path = rvc.run_inference(
        model_name=model_name,
        input_filename=input_filename,
        output_filename=output_filename,
        pitch=pitch,
        f0_method=f0_method,
        index_rate=index_rate,
        volume_norm=volume_norm,
        consonant_protect=consonant_protect
    )

# Example usage:
if __name__ == "__main__":
    run_voice_conversion(
        model_url=model_url,
        model_name=model_name,
        working_dir=working_dir,
        input_filename=input_audio,
        output_filename=output_audio,
        pitch=pitch,
        f0_method=f0_method,
        index_rate=index_rate,
        volume_norm=volume_normalization,
        consonant_protect=consonant_protection
    )